# NEX-GDDP-CMIP6 Yearly Indicator Counts (system A)

The climate-risk process needs, per model and year, **how many days crossed a threshold**:
heat days, ice days, tropical nights, frost days, dry days and heavy-rain days. Computing them
per request from the raw daily files is not feasible: each daily file stores one compressed
block for the whole globe, so a point query downloads every year file it touches (see
`docs/issues/climate-risk-yearly-aggregation.md`).

This notebook computes the counts **once**, with GeoEngine, and stores them as datasets:

* **One dataset per indicator x model x scenario**, named
  `nexgddp_cmip6_{model}_{scenario}_{indicator}_yearly`. Each is a single-band time series with
  one slice per calendar year.
* **The graph** is the daily threshold mask summed per calendar year:
  `MdGdalSource -> Expression -> TemporalRasterAggregation (sum, 1 year)`.
  `datasetFromProcessingGraph` writes one Cloud-Optimized GeoTIFF per year (ZSTD, internal
  mask) and a `dataset.json` into its upload folder, and registers a `MultiBandGdalSource`
  dataset. The export notebook packs that folder.
* **The thresholds live here**, in `INDICATORS`. The BioIS backend only knows the dataset
  names, so this table is the one place that defines what a heat day is.

Pipeline: `nexgddp_cmip6_ingest_*.ipynb` (raw daily datasets) -> **this notebook** (yearly
counts on system A) -> `geoengine_dataset_export.ipynb` -> `geoengine_dataset_import.ipynb`
(system B, where BioIS runs).

In [1]:
import datetime as dt
import json
import shutil
import time
from pathlib import Path

import geoengine as ge
import requests
from IPython.display import clear_output

try:
    import tomllib
except ModuleNotFoundError:  # Python < 3.11
    import tomli as tomllib

## Configuration

`DEV_ONLY` builds a small subset (one model, two scenarios, two indicators, the first years)
so a trial run stays short. The live backend test needs 5 years of heat and ice days for
ACCESS-CM2 `historical` and `ssp245`, which is exactly the dev subset.

In [2]:
GEOENGINE_URL = "http://localhost:3030/api"
GEOENGINE_USER = "admin@localhost"
GEOENGINE_PASSWORD = "adminadmin"

# Full model registry (id/variant/grid/scenarios). The BioIS registry is regenerated from it
# at the end, listing only what has all its yearly datasets.
REGISTRY_PATH = Path("../backend/conf/nexgddp_models.full.toml")
BIOIS_REGISTRY_PATH = Path("../backend/conf/nexgddp_models.toml")
# Spot-check values, carried to system B by the export notebook and re-checked after import.
SPOTCHECKS_PATH = Path("out/nexgddp_yearly_spotchecks.json")
# A's upload folder (`[upload] path` in its settings). Only used to report folders left behind
# by tasks that a server restart interrupted; None skips that check.
A_UPLOAD_ROOT = Path("/home/droenner/git/geoengine.worktrees/gdal-writer/geoengine/upload")

DEV_ONLY = False
DEV_MODELS = ["ACCESS-CM2"]
DEV_SCENARIOS = ["historical", "ssp245"]
DEV_INDICATORS = ["heatDays", "iceDays"]
DEV_YEARS = 5  # first years of each scenario

# GeoEngine runs each dataset as a background task. All indicators of one raw dataset (heat and
# ice days both read tasmax) start together and share its download. This many raw datasets are
# processed at once; each brings its two indicator tasks. Every task sends many small metadata
# queries to GeoEngine's database, whose connection pool has 10 connections: more than 2 raw
# datasets at once made tasks fail with "Timed out in bb8".
MAX_PARALLEL_RAW = 2
POLL_SECONDS = 10

# Tag that selects these datasets in the export notebook.
TAG = "nexgddp-yearly"

with open(REGISTRY_PATH, "rb") as fh:
    REGISTRY = tomllib.load(fh)["model"]

## Indicators

Each indicator is a daily threshold on one raw variable. Daily values `A` are temperatures in
Kelvin (`A - 273.15` gives °C) or precipitation in kg m-2 s-1 (`A * 86400` gives mm/day).
Thresholds follow the WMO climate-index definitions.

The expression yields 1 on an event day and 0 otherwise. Days without data stay no data
(`mapNoData: false`), and the yearly sum ignores them, so ocean cells stay no data.

In [3]:
INDICATORS = {
    "heatDays": {
        "source": "tasmax",
        "title": "Heat Days",
        "rule": "daily maximum temperature >= 30 °C",
        "expression": "let c = (A - 273.15); if c >= 30 { 1 } else { 0 }",
    },
    "iceDays": {
        "source": "tasmax",
        "title": "Ice Days",
        "rule": "daily maximum temperature < 0 °C",
        "expression": "let c = (A - 273.15); if c < 0 { 1 } else { 0 }",
    },
    "tropicalNights": {
        "source": "tasmin",
        "title": "Tropical Nights",
        "rule": "daily minimum temperature > 20 °C",
        "expression": "let c = (A - 273.15); if c > 20 { 1 } else { 0 }",
    },
    "frostDays": {
        "source": "tasmin",
        "title": "Frost Days",
        "rule": "daily minimum temperature < 0 °C",
        "expression": "let c = (A - 273.15); if c < 0 { 1 } else { 0 }",
    },
    "dryDays": {
        "source": "pr",
        "title": "Dry Days",
        "rule": "daily precipitation < 1 mm",
        "expression": "if (A * 86400) < 1 { 1 } else { 0 }",
    },
    "heavyRainDays": {
        "source": "pr",
        "title": "Heavy Rain Days",
        "rule": "daily precipitation > 20 mm",
        "expression": "if (A * 86400) > 20 { 1 } else { 0 }",
    },
}


def raw_name(model, scenario, source):
    return f"nexgddp_cmip6_{model}_{scenario}_{source}"


def yearly_name(model, scenario, indicator):
    return f"nexgddp_cmip6_{model}_{scenario}_{indicator}_yearly"

## Connect

All calls go through the plain REST API with the session token, so metadata stays exactly the
JSON the server sends.

In [4]:
ge.initialize(GEOENGINE_URL, credentials=(GEOENGINE_USER, GEOENGINE_PASSWORD))
http = requests.Session()
http.headers.update(ge.get_session().auth_header)


# A build runs for days; a short overload of the server (its database pool) must not end it.
RETRY_WAIT_SECONDS = 30
RETRY_FOR_SECONDS = 600


def api(method, path, **kwargs):
    # Call the GeoEngine API and return the JSON body; raise with the server's message.
    # Server errors and database timeouts are retried. Connection errors only for GET: a POST that
    # broke off may already have started a task, and sending it again would start it twice.
    deadline = time.time() + RETRY_FOR_SECONDS
    while True:
        try:
            response = http.request(method, f"{GEOENGINE_URL}/{path}", timeout=600, **kwargs)
            if response.ok:
                return response.json() if response.content else None
            error = f"{method} {path}: {response.status_code} {response.text}"
            if not (response.status_code >= 500 or (response.status_code == 401 and "Timeout" in response.text)):
                raise RuntimeError(error)
        except requests.ConnectionError as exc:
            if method != "GET":
                raise
            error = f"{method} {path}: {exc}"
        if time.time() > deadline:
            raise RuntimeError(f"giving up after {RETRY_FOR_SECONDS} s: {error}")
        print(f"{time.strftime('%H:%M:%S')} retrying in {RETRY_WAIT_SECONDS} s: {error[:200]}")
        time.sleep(RETRY_WAIT_SECONDS)


def dataset_or_none(name):
    response = http.get(f"{GEOENGINE_URL}/dataset/{name}", timeout=60)
    return response.json() if response.ok else None


def utc_ms(year):
    return int(dt.datetime(year, 1, 1, tzinfo=dt.timezone.utc).timestamp() * 1000)


def paged(path, **params):
    # All items of a paged listing; the server returns at most 20 per page.
    items, offset = [], 0
    while True:
        page = api("GET", path, params={**params, "offset": offset, "limit": 20})
        items += page
        if len(page) < 20:
            return items
        offset += 20


def running_tasks():
    # Description -> id of the tasks running on the server.
    return {t["description"]: t["taskId"] for t in paged("tasks/list", filter="running")}


print(api("GET", "info"))

{'buildDate': '2026-10-27', 'commitHash': '2555ac9cc', 'version': '0.10.0', 'features': ''}


## Targets

One target per (model, scenario, indicator) whose raw daily dataset is registered. The years
come from the raw dataset: its slices are stamped at noon, so its time bounds run from noon of
the first January 1st to noon of the January 1st after the last year.

The indicators of one raw dataset (heat and ice days both read `tasmax`) are built together:
tasks started at the same time share the raw tiles through GeoEngine's cache, so the raw data is
downloaded once. Measured for two indicators of one year: 272 MB received in total, about the
size of one year file. One after the other does not work for a full scenario, because the first
task's 86 years of raw tiles would no longer fit in the cache when the second one starts.

In [5]:
def raw_years(name):
    # Whole calendar years of a raw daily dataset, or None if it is not registered.
    info = dataset_or_none(name)
    if info is None:
        return None
    bounds = info["resultDescriptor"]["time"]["bounds"]
    first = dt.datetime.fromtimestamp(bounds["start"] / 1000, dt.timezone.utc).year
    last = dt.datetime.fromtimestamp(bounds["end"] / 1000, dt.timezone.utc).year - 1
    return first, last


targets = []
for model in REGISTRY:
    if DEV_ONLY and model["id"] not in DEV_MODELS:
        continue
    for scenario in model["scenarios"]:
        if DEV_ONLY and scenario not in DEV_SCENARIOS:
            continue
        for indicator, spec in INDICATORS.items():
            if DEV_ONLY and indicator not in DEV_INDICATORS:
                continue
            raw = raw_name(model["id"], scenario, spec["source"])
            years = raw_years(raw)
            if years is None:
                print(f"skip {raw}: raw dataset not registered")
                continue
            if DEV_ONLY:
                years = (years[0], min(years[1], years[0] + DEV_YEARS - 1))
            targets.append({
                "model": model["id"],
                "scenario": scenario,
                "indicator": indicator,
                "raw": raw,
                "years": years,
                "name": yearly_name(model["id"], scenario, indicator),
            })

targets.sort(key=lambda t: (t["model"], t["scenario"], INDICATORS[t["indicator"]]["source"], t["indicator"]))
for t in targets:
    print(f"{t['name']}: {t['years'][0]}-{t['years'][1]}")

nexgddp_cmip6_ACCESS-CM2_historical_dryDays_yearly: 1950-2014
nexgddp_cmip6_ACCESS-CM2_historical_heavyRainDays_yearly: 1950-2014
nexgddp_cmip6_ACCESS-CM2_historical_heatDays_yearly: 1950-2014
nexgddp_cmip6_ACCESS-CM2_historical_iceDays_yearly: 1950-2014
nexgddp_cmip6_ACCESS-CM2_historical_frostDays_yearly: 1950-2014
nexgddp_cmip6_ACCESS-CM2_historical_tropicalNights_yearly: 1950-2014
nexgddp_cmip6_ACCESS-CM2_ssp245_dryDays_yearly: 2015-2100
nexgddp_cmip6_ACCESS-CM2_ssp245_heavyRainDays_yearly: 2015-2100
nexgddp_cmip6_ACCESS-CM2_ssp245_heatDays_yearly: 2015-2100
nexgddp_cmip6_ACCESS-CM2_ssp245_iceDays_yearly: 2015-2100
nexgddp_cmip6_ACCESS-CM2_ssp245_frostDays_yearly: 2015-2100
nexgddp_cmip6_ACCESS-CM2_ssp245_tropicalNights_yearly: 2015-2100
nexgddp_cmip6_ACCESS-CM2_ssp370_dryDays_yearly: 2015-2100
nexgddp_cmip6_ACCESS-CM2_ssp370_heavyRainDays_yearly: 2015-2100
nexgddp_cmip6_ACCESS-CM2_ssp370_heatDays_yearly: 2015-2100
nexgddp_cmip6_ACCESS-CM2_ssp370_iceDays_yearly: 2015-2100
nexgddp_c

## The yearly graph

The daily mask is `I8` (0/1) and the yearly sum `U16`: counts reach 366, too many for 8 bits.
The 1-year window has no reference, so windows are calendar years.

In [6]:
def yearly_operator(raw, indicator):
    return {
        "type": "TemporalRasterAggregation",
        "params": {
            "aggregation": {"type": "sum", "ignoreNoData": True},
            "window": {"granularity": "years", "step": 1},
            "outputType": "U16",
        },
        "sources": {
            "raster": {
                "type": "Expression",
                "params": {
                    "expression": INDICATORS[indicator]["expression"],
                    "outputType": "I8",
                    "outputBand": {
                        "name": indicator,
                        "measurement": {"type": "continuous", "measurement": "Days", "unit": "Days"},
                    },
                    "mapNoData": False,
                },
                "sources": {"raster": {"type": "MdGdalSource", "params": {"data": raw}}},
            }
        },
    }


GLOBAL_BOUNDS = {
    "upperLeftCoordinate": {"x": -180.0, "y": 90.0},
    "lowerRightCoordinate": {"x": 180.0, "y": -60.0},
}

## Leftovers of interrupted tasks

A task writes its files into a new upload folder and registers the dataset only at the end. After
a server restart, the folders of interrupted tasks stay behind without a dataset. This cell lists
upload folders that no dataset references, with size and last change. It only reports: to delete
folders, copy their ids into `DELETE_LEFTOVERS` and run the cell again.

While tasks are running the check is skipped, because their folders have no dataset yet either.
Uploads made for other purposes (e.g. files uploaded through the API) also show up here, so
check each id before deleting it.

In [7]:
DELETE_LEFTOVERS = []  # upload ids from the report below

if A_UPLOAD_ROOT is None or not A_UPLOAD_ROOT.is_dir():
    print(f"upload folder {A_UPLOAD_ROOT} not available, check skipped")
elif running_tasks():
    print("tasks are running on the server, check skipped")
else:
    # dataPath is {"upload": id}, {"volume": name} or the string "external"
    data_paths = [api("GET", f"dataset/{d['name']}").get("dataPath") for d in paged("datasets", order="NameAsc")]
    referenced = {p["upload"] for p in data_paths if isinstance(p, dict) and "upload" in p}
    leftovers = {p.name: p for p in A_UPLOAD_ROOT.iterdir() if p.is_dir() and p.name not in referenced}
    for upload, folder in sorted(leftovers.items(), key=lambda item: item[1].stat().st_mtime):
        files = [f for f in folder.rglob("*") if f.is_file()]
        changed = dt.datetime.fromtimestamp(folder.stat().st_mtime)
        print(f"{upload}: {len(files)} files, {sum(f.stat().st_size for f in files) / 1e6:.1f} MB, "
              f"last change {changed:%Y-%m-%d %H:%M}, dataset.json: {(folder / 'dataset.json').exists()}")
    print(f"{len(leftovers)} upload folders without dataset")
    for upload in DELETE_LEFTOVERS:
        assert upload in leftovers, f"{upload} is not a leftover"
        shutil.rmtree(leftovers[upload])
        print(f"deleted {upload}")

d0eb7346-afe8-4692-8d98-876b7fcdfd91: 2 files, 0.1 MB, last change 2026-10-23 00:23, dataset.json: True
ab958dbe-18be-4053-9713-977318e1c1c8: 2 files, 1.4 MB, last change 2026-10-23 00:23, dataset.json: True
daacbde4-35d8-4aa8-b2bc-5fda68104440: 2 files, 0.1 MB, last change 2026-10-23 00:24, dataset.json: True
9bd98697-7541-44b3-9653-d6c3bd1ef61e: 2 files, 0.1 MB, last change 2026-10-23 00:24, dataset.json: True
a56bf6ca-defb-4ac3-bd22-5c21504a6fe9: 2 files, 0.1 MB, last change 2026-10-23 00:25, dataset.json: True
901d11e9-2731-42d1-8ff0-142e2d5cf33e: 0 files, 0.0 MB, last change 2026-10-23 17:57, dataset.json: False
563cbc80-06c7-4f3d-b3ee-9c394ca0ff25: 0 files, 0.0 MB, last change 2026-10-23 17:57, dataset.json: False
303bc365-e42e-40d8-b5e4-59c3ae2e0339: 0 files, 0.0 MB, last change 2026-10-23 17:58, dataset.json: False
25ea7a0a-6dd6-4bf4-985f-f77c76e6fe5e: 0 files, 0.0 MB, last change 2026-10-23 17:58, dataset.json: False
3c0b1a81-5965-4ef2-b9b0-368acc76d201: 3 files, 0.5 MB, last 

## Metadata

`datasetFromProcessingGraph` leaves symbology and provenance empty and tags the dataset
`workflow`. Everything a reader on system B needs goes into the metadata, because the metadata
is what the export notebook carries over. `set_metadata` is applied to each dataset as soon as its
task completes (in the build loop below), and here to every dataset built so far, so a re-run also
repairs datasets of a run that stopped early:

* tags that select and describe the dataset (`nexgddp-yearly`, `indicator:...`, ...),
* the source citation as provenance,
* a symbology for 0-366 days,
* read permission for registered and anonymous users.

In [8]:
def description(target):
    spec = INDICATORS[target["indicator"]]
    return (
        f"Days per calendar year with {spec['rule']}, from NEX-GDDP-CMIP6 {target['model']} "
        f"{target['scenario']} ({spec['source']}). Expression: `{spec['expression']}`, summed per "
        f"year. Built by notebooks/nexgddp_cmip6_aggregate_yearly.ipynb."
    )


def display_name(target):
    return (f"NEX-GDDP-CMIP6 {INDICATORS[target['indicator']]['title']} per year "
            f"({target['model']}, {target['scenario']})")


PROVENANCE = [{
    "citation": "Thrasher, B. et al. NASA Global Daily Downscaled Projections, CMIP6. "
                "Scientific Data 9, 262 (2022). https://doi.org/10.1038/s41597-022-01393-4",
    "license": "Per-model CMIP6 licenses, see the NEX-GDDP-CMIP6 technical note",
    "uri": "https://www.nccs.nasa.gov/services/data-collections/land-based-products/nex-gddp-cmip6",
}]
SYMBOLOGY = ge.RasterSymbology(
    raster_colorizer=ge.SingleBandRasterColorizer(
        band=0,
        band_colorizer=ge.Colorizer.linear_with_mpl_cmap(color_map="YlOrRd", min_max=(0, 366), n_steps=10),
    ),
).to_api_dict().to_dict()
READ_ROLES = [ge.permissions.REGISTERED_USER_ROLE_ID, ge.permissions.ANONYMOUS_USER_ROLE_ID]


def set_metadata(target):
    # Tags, provenance, symbology and read permission: datasetFromProcessingGraph sets none of them,
    # and without the permission only the admin can read the dataset.
    name = target["name"]
    info = dataset_or_none(name)
    api("POST", f"dataset/{name}", json={
        "name": name,
        "display_name": info["displayName"],
        "description": description(target),
        "tags": [TAG, f"indicator:{target['indicator']}", f"model:{target['model']}",
                 f"scenario:{target['scenario']}", f"source:{INDICATORS[target['indicator']]['source']}"],
    })
    api("PUT", f"dataset/{name}/provenance", json={"provenances": PROVENANCE})
    api("PUT", f"dataset/{name}/symbology", json=SYMBOLOGY)
    for role in READ_ROLES:
        api("PUT", "permissions", json={
            "resource": {"type": "dataset", "id": name},
            "roleId": str(role),
            "permission": "Read",
        })


# datasets built by earlier runs, which may have stopped before setting their metadata
built = [t for t in targets if dataset_or_none(t["name"]) is not None]
for target in built:
    set_metadata(target)
print(f"metadata set on {len(built)} built datasets")

metadata set on 120 built datasets


## Materialize

The cell can be re-run after an interruption:

* Datasets that already exist are skipped.
* Stopping the kernel does not stop the tasks on the server. A re-run takes over the tasks of its
  targets that are still running (matched by the dataset's display name in the task
  description) and waits for them instead of submitting them again. If only part of a raw
  dataset's indicators is still running, the others start at once, so they still share the
  download.
* A server restart ends all running tasks. Their datasets do not exist, so a re-run builds them
  again from their first year: a dataset is registered only when its whole time series is
  written.
* Short server overloads do not end the run: API calls are retried (see `api`), and a task that
  fails for a transient reason (`TRANSIENT_ERRORS`: GeoEngine's database pool timed out, or a
  read of a raw S3 file broke off) is submitted again, up to `MAX_ATTEMPTS` times. Other failures
  are reported and stop the run at the end of the cell.

In [9]:
def submit(target):
    graph = {"type": "Raster", "operator": yearly_operator(target["raw"], target["indicator"])}
    graph_id = api("POST", "processingGraphs", json=graph)["id"]
    first, last = target["years"]
    return api("POST", f"datasetFromProcessingGraph/{graph_id}", json={
        "name": target["name"],
        "displayName": display_name(target),
        "description": description(target),
        "query": {"spatialBounds": GLOBAL_BOUNDS,
                  "timeInterval": {"start": utc_ms(first), "end": utc_ms(last + 1)}},
        "asCog": True,
    })["taskId"]


# one group per raw dataset: its indicators that are not built yet
groups = {}
for t in targets:
    if dataset_or_none(t["name"]) is None:
        groups.setdefault(t["raw"], []).append(t)
pending = list(groups.values())
total = sum(map(len, pending))
events = [f"{len(targets) - total} already built, {total} to build from {len(pending)} raw datasets"]
running, failed, built_minutes = {}, [], []
attempts = {}  # dataset name -> number of submissions
MAX_ATTEMPTS = 3
# Failures worth another attempt: the database pool was exhausted, or a read of a raw file from S3
# broke off. The file itself was fine in both cases seen ("not recognized" after a bad read).
TRANSIENT_ERRORS = ("Timed out in bb8", "CURL error", "not recognized as being in a supported file format")


def show_status():
    # Redraw the status: GeoEngine reports no progress for these tasks, so show elapsed times.
    clear_output(wait=True)
    waiting = sum(map(len, pending))
    lines = [f"{time.strftime('%H:%M:%S')}  built {len(built_minutes)}/{total}, running {len(running)}, "
             f"waiting {waiting}, failed {len(failed)}"]
    if built_minutes and (waiting or running):
        # tasks of a group run together, MAX_PARALLEL_RAW groups at once
        per_group = sum(built_minutes) / len(built_minutes)
        groups_left = len(pending) + len({t["raw"] for t, _ in running.values()})
        lines.append(f"  about {groups_left / MAX_PARALLEL_RAW * per_group / 60:.1f} h left "
                     f"({per_group:.1f} min per raw dataset so far)")
    for target, started in sorted(running.values(), key=lambda item: item[1]):
        lines.append(f"  running {(time.time() - started) / 60:6.1f} min  {target['name']}")
    lines += [""] + events[-15:]
    print("\n".join(lines))

# take over tasks of these targets that are still running on the server
on_server = running_tasks()
for group in pending:
    taken_over = False
    for target in list(group):
        prefix = f"Creating dataset {display_name(target)} from "
        task_id = next((i for d, i in on_server.items() if d.startswith(prefix)), None)
        if task_id is not None:
            running[task_id] = (target, time.time())
            group.remove(target)
            taken_over = True
            events.append(f"taken over: {target['name']} (task {task_id})")
    if taken_over:
        # the rest of the group joins the running tasks to share their download
        while group:
            target = group.pop(0)
            running[submit(target)] = (target, time.time())
pending = [g for g in pending if g]

while pending or running:
    while pending and len({t["raw"] for t, _ in running.values()}) < MAX_PARALLEL_RAW:
        for target in pending.pop(0):
            running[submit(target)] = (target, time.time())
    show_status()
    time.sleep(POLL_SECONDS)
    for task_id, (target, started) in list(running.items()):
        status = api("GET", f"tasks/{task_id}/status")
        if status["status"] == "running":
            continue
        del running[task_id]
        minutes = (time.time() - started) / 60
        stamp = time.strftime("%H:%M")
        if status["status"] == "completed":
            set_metadata(target)
            built_minutes.append(minutes)
            events.append(f"{stamp} built  {target['name']} in {minutes:.1f} min")
        else:
            error = json.dumps(status.get("error"))
            attempts[target["name"]] = attempts.get(target["name"], 1) + 1
            if any(e in error for e in TRANSIENT_ERRORS) and attempts[target["name"]] <= MAX_ATTEMPTS:
                running[submit(target)] = (target, time.time())
                events.append(f"{stamp} RETRY  {target['name']} after {minutes:.1f} min: {error[:120]}")
            else:
                failed.append(target["name"])
                events.append(f"{stamp} FAILED {target['name']} after {minutes:.1f} min: {error}")
show_status()

assert not failed, f"{len(failed)} datasets failed: {failed}"

23:17:28  built 0/0, running 0, waiting 0, failed 0

120 already built, 0 to build from 0 raw datasets


## Verify

For every built dataset:

1. **Shape:** a `MultiBandGdalSource` with one `U16` band named after the indicator, and time
   bounds of exactly the built years.
2. **Values:** at three land points and the first and last year, the materialized count equals
   the count the raw graph computes on the fly, and lies within 0-366. An ocean point is no data.

The materialized values are written to `SPOTCHECKS_PATH`, so the import notebook can check
that system B returns the same numbers. A full run spot-checks one dataset per indicator,
because each on-the-fly check reads raw daily files.

In [10]:
LAND_POINTS = {"Berlin": (13.405, 52.52), "Madrid": (-3.70, 40.42), "Delhi": (77.21, 28.61)}
OCEAN_POINT = (-30.0, 30.0)  # mid Atlantic


def point_values(operator, lon, lat, first, last):
    # Yearly values of a raster operator at a point, via WFS on a RasterVectorJoin.
    graph = {"type": "Vector", "operator": {
        "type": "RasterVectorJoin",
        "params": {
            "names": {"type": "names", "values": ["value"]},
            "featureAggregation": "first",
            "temporalAggregation": "none",
        },
        "sources": {
            "vector": {"type": "MockPointSource", "params": {"points": [{"x": lon, "y": lat}]}},
            "rasters": [operator],
        },
    }}
    graph_id = api("POST", "processingGraphs", json=graph)["id"]
    d = 0.0001
    collection = api("GET", f"wfs/{graph_id}", params={
        "service": "WFS", "version": "2.0.0", "request": "GetFeature", "typeNames": graph_id,
        "bbox": f"{lon - d},{lat - d},{lon + d},{lat + d}", "srsName": "EPSG:4326",
        "time": f"{first}-01-01T00:00:00Z/{last + 1}-01-01T00:00:00Z",
    })
    return {
        int(f["when"]["start"][:4]): f["properties"]["value"]
        for f in collection["features"]
    }


built = [t for t in targets if dataset_or_none(t["name"]) is not None]
spot_targets = targets if DEV_ONLY else list({t["indicator"]: t for t in built}.values())
spotchecks = []
for target in built:
    name = target["name"]
    info = dataset_or_none(name)
    rd = info["resultDescriptor"]
    first, last = target["years"]
    assert info["sourceOperator"] == "MultiBandGdalSource", info["sourceOperator"]
    assert rd["dataType"] == "U16" and [b["name"] for b in rd["bands"]] == [target["indicator"]], rd
    bounds = rd["time"]["bounds"]
    assert (bounds["start"], bounds["end"]) == (utc_ms(first), utc_ms(last + 1)), f"{name}: {bounds}"

    if target not in spot_targets:
        continue
    stored_op = {"type": "MultiBandGdalSource", "params": {"data": name}}
    raw_op = yearly_operator(target["raw"], target["indicator"])
    for place, (lon, lat) in LAND_POINTS.items():
        stored = point_values(stored_op, lon, lat, first, last)
        assert sorted(stored) == list(range(first, last + 1)), f"{name} {place}: years {sorted(stored)}"
        computed = point_values(raw_op, lon, lat, first, first)
        computed.update(point_values(raw_op, lon, lat, last, last))
        for year in sorted({first, last}):
            assert stored[year] == computed[year], f"{name} {place} {year}: {stored[year]} != {computed[year]}"
            assert 0 <= stored[year] <= 366, f"{name} {place} {year}: {stored[year]}"
            spotchecks.append({"dataset": name, "lon": lon, "lat": lat, "year": year, "value": stored[year]})
    ocean = point_values(stored_op, *OCEAN_POINT, first, first)
    assert ocean.get(first) is None, f"{name}: ocean value {ocean}"
    print(f"verified {name}")

SPOTCHECKS_PATH.parent.mkdir(exist_ok=True)
SPOTCHECKS_PATH.write_text(json.dumps(spotchecks, indent=1))
print(f"{len(spotchecks)} spot checks written to {SPOTCHECKS_PATH}")

verified nexgddp_cmip6_MRI-ESM2-0_ssp585_dryDays_yearly


verified nexgddp_cmip6_MRI-ESM2-0_ssp585_heavyRainDays_yearly


verified nexgddp_cmip6_MRI-ESM2-0_ssp585_heatDays_yearly


verified nexgddp_cmip6_MRI-ESM2-0_ssp585_iceDays_yearly


verified nexgddp_cmip6_MRI-ESM2-0_ssp585_frostDays_yearly


verified nexgddp_cmip6_MRI-ESM2-0_ssp585_tropicalNights_yearly
36 spot checks written to out/nexgddp_yearly_spotchecks.json


## BioIS model registry

The backend offers a model/scenario only if it is listed in `backend/conf/nexgddp_models.toml`,
and it needs all six indicators for every listed pair. The registry is rewritten from the full
registry, keeping only pairs whose six yearly datasets exist. A `DEV_ONLY` run builds two
indicators, so it only prints what a full run would write.

In [11]:
entries = []
for model in REGISTRY:
    scenarios = [
        s for s in model["scenarios"]
        if all(dataset_or_none(yearly_name(model["id"], s, i)) is not None for i in INDICATORS)
    ]
    # anomalies need the historical reference, so a model without it is left out
    if "historical" in scenarios and len(scenarios) > 1:
        entries.append({**model, "scenarios": scenarios})

header = (
    "# NEX-GDDP-CMIP6 model registry - the source of truth for what climate-risk can compute.\n"
    "#\n"
    "# Every model/scenario listed here has all six yearly indicator datasets\n"
    "# (`nexgddp_cmip6_{model}_{scenario}_{indicator}_yearly`), so the process description and\n"
    "# the `models` dropdown offer exactly what can run. This file is rewritten from\n"
    "# `nexgddp_models.full.toml` by `notebooks/nexgddp_cmip6_aggregate_yearly.ipynb`.\n"
)
body = "".join(
    f'\n[[model]]\nid = "{m["id"]}"\nvariant = "{m["variant"]}"\ngrid = "{m["grid"]}"\n'
    f"scenarios = {json.dumps(m['scenarios'])}\n"
    for m in entries
)
if DEV_ONLY:
    print(f"DEV_ONLY: would list {len(entries)} models:\n{body or '(none)'}")
else:
    BIOIS_REGISTRY_PATH.write_text(header + body)
    print(f"{BIOIS_REGISTRY_PATH} lists {len(entries)} models")

../backend/conf/nexgddp_models.toml lists 5 models


## Next

Export the datasets with `geoengine_dataset_export.ipynb` (`SELECT_TAG = "nexgddp-yearly"`)
and import them on system B with `geoengine_dataset_import.ipynb`.